# Notebook 02 — Hypothesis-Driven Exploratory Data Analysis

**Project:** Telco Customer Churn Analysis

This notebook investigates statistically supported relationships between
customer attributes and churn. Rather than generating plots and "finding
insights" after the fact, every analysis here starts from a pre-defined,
business-relevant hypothesis, states the null hypothesis, selects an
appropriate statistical test, and reports the *actual* test statistic,
p-value, and effect size computed from the cleaned dataset.

**Grounding principle:** every number reported in this notebook is computed at
run time from `data/cleaned/telco_churn_clean.csv`. No analytical result is
hard-coded or inherited from earlier notebooks.

**Prerequisite:** Notebook 01 (`01_data_cleaning.ipynb`) produced the cleaned
dataset used here. This notebook performs no further cleaning and makes no
modeling decisions — predictive modeling is deliberately deferred to Notebook 03.

## 2. Imports

A lightweight scientific Python stack is used throughout. `scipy` provides the
statistical tests, `statsmodels` provides the multiple-comparisons correction,
and `pandas`/`numpy`/`matplotlib`/`seaborn` handle data and visualization.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, mannwhitneyu
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

## 3. Load the Cleaned Dataset

Load `data/cleaned/telco_churn_clean.csv` and validate it **before** any
analysis. The path is resolved robustly so the notebook runs whether the
kernel starts from the repository root or from the `notebooks/` directory.

Validation expectations (set by Notebook 01):
- 7,032 rows (after dropping 11 rows with missing `TotalCharges`)
- a `Churn` column containing only the values 0 and 1

If any expectation fails, the problem is raised loudly instead of being
silently patched.

In [2]:
DATA_FILE = "data/cleaned/telco_churn_clean.csv"


def find_data_path() -> Path:
    """Locate the cleaned dataset whether the kernel runs from the
    repository root or from the notebooks/ directory."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        candidate = folder / DATA_FILE
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate {DATA_FILE}. Run this notebook from the repository "
        "root or from the notebooks/ directory."
    )


DATA_PATH = find_data_path()
print("Resolved data path:", DATA_PATH)

Resolved data path: /Users/vaibhavvikasranjan/Downloads/telco-churn-analysis/data/cleaned/telco_churn_clean.csv


In [3]:
EXPECTED_ROWS = 7032

df = pd.read_csv(DATA_PATH)
print("Loaded shape:", df.shape)

problems = []
if df.shape[0] != EXPECTED_ROWS:
    problems.append(f"Expected {EXPECTED_ROWS} rows, found {df.shape[0]}")
if "Churn" not in df.columns:
    problems.append("Churn column is missing")
elif not set(df["Churn"].unique()).issubset({0, 1}):
    problems.append(
        f"Churn contains unexpected values: {sorted(df['Churn'].unique())}"
    )

if problems:
    raise ValueError("Dataset validation failed:\n- " + "\n- ".join(problems))

missing = df.isna().sum()
duplicated_rows = int(df.duplicated().sum())

print("Missing values per column:")
print(missing[missing > 0].to_string() if (missing > 0).any() else "  None")
print("Duplicate rows:", duplicated_rows)
print()
print("Target (Churn) distribution:")
print(df["Churn"].value_counts().to_string())
print()
print(f"Validation passed: {df.shape[0]} rows x {df.shape[1]} columns, "
      f"Churn in {{0, 1}}.")

Loaded shape: (7032, 20)
Missing values per column:
  None
Duplicate rows: 22

Target (Churn) distribution:
Churn
0    5163
1    1869

Validation passed: 7032 rows x 20 columns, Churn in {0, 1}.


## 4. Baseline Churn Analysis

Before testing any hypothesis, establish the baseline: how many customers are
in the dataset, how many churned, and the exact churn rate. All values are
computed dynamically from the loaded data.

In [4]:
total_customers = len(df)
churned_customers = int((df["Churn"] == 1).sum())
retained_customers = int((df["Churn"] == 0).sum())
churn_rate = churned_customers / total_customers
retention_rate = retained_customers / total_customers

print(f"Total customers:  {total_customers}")
print(f"Churned:          {churned_customers}")
print(f"Retained:         {retained_customers}")
print(f"Churn rate:       {churn_rate:.4f} ({churn_rate:.2%})")
print(f"Retention rate:   {retention_rate:.4f} ({retention_rate:.2%})")

Total customers:  7032
Churned:          1869
Retained:         5163
Churn rate:       0.2658 (26.58%)
Retention rate:   0.7342 (73.42%)


In [5]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df["Churn"].value_counts().reindex([0, 1])
ax.bar(["Retained (0)", "Churned (1)"], counts.values,
       color=["#1f77b4", "#d62728"], width=0.55)
ax.set_ylabel("Number of customers")
ax.set_title("Churn Distribution")
for i, v in enumerate(counts.values):
    ax.text(i, v + 30, f"{v}", ha="center")
plt.tight_layout()
plt.show()

## 5. Reusable Statistical Helper Functions

Two test families are used throughout, depending on the data type of the
feature being compared with the binary `Churn` target:

- **Categorical feature vs Churn** → chi-square test of independence. The test
  compares observed cell counts against counts expected under independence.
  The effect size is **Cramér's V** (0 = no association, 1 = perfect
  association).
- **Continuous feature vs Churn** → Mann-Whitney U test. This non-parametric
  test compares whether one group tends to have larger values than the other,
  and imposes **no normality assumption** on the feature. The effect size is
  the **rank-biserial correlation**, computed with the churned group as sample
  1: a positive value means the churned group tends to have *smaller* values,
  a negative value means it tends to have *larger* values.

A third helper summarises customer count, churn count, and churn rate per
category, so rates are always computed consistently.

In [6]:
def category_summary(df, feature, target="Churn"):
    """Customer count, churn count, and churn rate per category."""
    summary = (
        df.groupby(feature, observed=True)
        .agg(customers=(target, "size"), churned=(target, "sum"))
        .assign(churn_rate=lambda d: d["churned"] / d["customers"])
    )
    return summary


def chi_square_test(df, feature, target="Churn"):
    """Chi-square test of independence between a categorical feature and Churn.

    Returns a (result dict, contingency table) pair. The result holds the
    chi-square statistic, degrees of freedom, raw p-value, and Cramér's V.
    """
    contingency = pd.crosstab(df[feature], df[target])
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    n = contingency.values.sum()
    min_dim = min(contingency.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else np.nan

    result = {
        "feature": feature,
        "test": "Chi-square",
        "statistic": chi2,
        "degrees_of_freedom": dof,
        "p_value": p_value,
        "effect_size": cramers_v,
        "effect_size_measure": "Cramer's V",
    }
    return result, contingency


def mann_whitney_test(df, feature, target="Churn"):
    """Mann-Whitney U test between a continuous feature and binary Churn.

    No normality assumption is imposed. Effect size is the rank-biserial
    correlation r = 1 - 2*U/(n1*n2) with the churned group as sample 1, so a
    positive r means churned values tend to be smaller and a negative r means
    they tend to be larger.
    """
    churned = df.loc[df[target] == 1, feature]
    retained = df.loc[df[target] == 0, feature]
    u_stat, p_value = mannwhitneyu(churned, retained, alternative="two-sided")
    n1, n0 = len(churned), len(retained)
    rank_biserial = 1 - (2 * u_stat) / (n1 * n0)

    result = {
        "feature": feature,
        "test": "Mann-Whitney U",
        "statistic": u_stat,
        "degrees_of_freedom": np.nan,
        "p_value": p_value,
        "effect_size": rank_biserial,
        "effect_size_measure": "rank-biserial r",
        "n_churned": n1,
        "n_retained": n0,
        "median_churned": churned.median(),
        "median_retained": retained.median(),
    }
    return result


def plot_churn_rate_by_category(summary, feature, title):
    """Bar chart of churn rate (%) per category, annotated with counts."""
    plot_df = summary.sort_values("churn_rate", ascending=False).copy()
    plot_df["churn_rate_pct"] = plot_df["churn_rate"] * 100
    ax = plot_df["churn_rate_pct"].plot(
        kind="bar", figsize=(8, 4.5), color="#1f77b4", width=0.7
    )
    ax.set_ylabel("Churn rate (%)")
    ax.set_xlabel(feature)
    ax.set_title(title)
    for i, (rate, count) in enumerate(
        zip(plot_df["churn_rate_pct"], plot_df["customers"])
    ):
        ax.text(i, rate + 0.6, f"{rate:.1f}%", ha="center", fontsize=9)
        ax.text(i, -ax.get_ylim()[1] * 0.04, f"n={count}", ha="center",
                fontsize=8, color="dimgrey")
    ax.set_ylim(0, plot_df["churn_rate_pct"].max() * 1.18)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    plt.tight_layout()
    plt.show()


def plot_continuous_by_churn(df, feature, title):
    """Overlaid KDE of a continuous feature split by churn status.

    A KDE is used (rather than arbitrary binned groups) so the raw distribution
    shape is shown without imposing bucket boundaries on the analysis.
    """
    fig, ax = plt.subplots(figsize=(8, 4.5))
    sns.kdeplot(df.loc[df["Churn"] == 0, feature], label="Retained",
                color="#1f77b4", fill=True, alpha=0.35, ax=ax)
    sns.kdeplot(df.loc[df["Churn"] == 1, feature], label="Churned",
                color="#d62728", fill=True, alpha=0.35, ax=ax)
    ax.set_xlabel(feature)
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Hypothesis 1 — Contract Type

**Business hypothesis.** Month-to-month customers have a higher observed churn
rate than customers on one-year or two-year contracts.

**Null hypothesis (H0).** Contract type and churn are independent in the
population; any difference in churn rates across contract types is due to
chance.

**Why a chi-square test of independence?** Both `Contract` and `Churn` are
categorical variables. The chi-square test compares the observed contingency
table to the table expected under independence and makes no normality
assumption. Cramér's V quantifies the strength of the association.

In [7]:
res_contract, ct_contract = chi_square_test(df, "Contract")

display(category_summary(df, "Contract"))
print("\nContingency table (rows = Contract, columns = Churn):")
display(ct_contract)

print(
    f"chi-square = {res_contract['statistic']:.4f} | "
    f"df = {res_contract['degrees_of_freedom']} | "
    f"p = {res_contract['p_value']:.4g} | "
    f"Cramer's V = {res_contract['effect_size']:.4f}"
)

plot_churn_rate_by_category(
    category_summary(df, "Contract"),
    "Contract",
    "Churn Rate by Contract Type",
)

,customers,churned,churn_rate
Contract,,,
Month-to-month,3875,1655,0.427097
One year,1472,166,0.112772
Two year,1685,48,0.028487



Contingency table (rows = Contract, columns = Churn):


Churn,0,1
Contract,,
Month-to-month,2220,1655
One year,1306,166
Two year,1637,48


chi-square = 1179.5458 | df = 2 | p = 7.326e-257 | Cramer's V = 0.4096


### Result (computed from data)

Contract type was **statistically associated** with churn:

| Measure | Value |
|---|---|
| Chi-square statistic | 1179.55 |
| Degrees of freedom | 2 |
| p-value | 7.33e-257 |
| Cramér's V | 0.410 |

Observed churn rates by contract type:

| Contract | Customers | Churn rate |
|---|---|---|
| Month-to-month | 3,875 | 42.7% |
| One year | 1,472 | 11.3% |
| Two year | 1,685 | 2.8% |

**Plain-language interpretation.** Month-to-month customers churn at roughly
3.8x the rate of one-year customers and about 15x the rate of two-year
customers. Cramér's V ≈ 0.41 indicates a moderate-to-strong association,
well above the conventional 0.3 threshold often used for a medium effect.

**Association, not causation.** Customers are not randomly assigned to
contract types. Longer contracts may attract customers who already intend to
stay, or they may reduce churn by raising switching costs. This observational
analysis cannot separate those mechanisms, so we only say: contract type was
statistically associated with churn.

## 7. Hypothesis 2 — Tenure

**Business hypothesis.** Customers who churn have a different tenure
distribution from customers who remain.

**Null hypothesis (H0).** The distribution of tenure is the same for churned
and retained customers.

**Why Mann-Whitney U?** Tenure is a right-skewed, bounded count (0–72 months)
and no normality assumption should be imposed. Mann-Whitney U is a
non-parametric test of stochastic ordering between two groups. The effect size
is the rank-biserial correlation. Tenure is shown with a KDE rather than
arbitrary bins so the distribution is not pre-judged.

In [8]:
res_tenure = mann_whitney_test(df, "tenure")

print(f"Churned  group: n = {res_tenure['n_churned']},  "
      f"median tenure = {res_tenure['median_churned']}")
print(f"Retained group: n = {res_tenure['n_retained']}, "
      f"median tenure = {res_tenure['median_retained']}")
print(
    f"U = {res_tenure['statistic']:.4g} | "
    f"p = {res_tenure['p_value']:.4g} | "
    f"rank-biserial r = {res_tenure['effect_size']:.4f}"
)

plot_continuous_by_churn(df, "tenure", "Tenure Distribution by Churn Status")

Churned  group: n = 1869,  median tenure = 10.0
Retained group: n = 5163, median tenure = 38.0
U = 2.495e+06 | p = 6.043e-211 | rank-biserial r = 0.4829


### Result (computed from data)

Churned and retained customers had **significantly different tenure
distributions**:

| Measure | Value |
|---|---|
| Mann-Whitney U statistic | 2,494,979 |
| p-value | 6.04e-211 |
| Rank-biserial r | +0.483 |
| Median tenure, churned | 10 months |
| Median tenure, retained | 38 months |

**Plain-language interpretation.** The churned group's median tenure (10
months) is far below the retained group's (38 months). The positive
rank-biserial r (+0.48) means churned customers tended to have smaller tenure
values; this is a moderate-to-strong effect. In business terms, churn risk is
heavily concentrated among relatively new customers.

**Association, not causation.** Tenure is a consequence of a customer's
history, not a treatment that can be assigned. Low tenure and churn may both
reflect an early "fit" problem with the service; we cannot claim that being a
customer for a shorter time *causes* churn.

## 8. Hypothesis 3 — MonthlyCharges

**Business hypothesis.** Customers who churn have a different distribution of
monthly charges from customers who remain.

**Null hypothesis (H0).** The distribution of MonthlyCharges is the same for
churned and retained customers.

**Why Mann-Whitney U?** Monthly charges are skewed (bounded below at $0, with
price points tied to service tiers), so the normality assumption of a t-test
is not appropriate. Mann-Whitney U compares the two distributions
non-parametrically; the rank-biserial correlation is reported as the effect
size.

In [9]:
res_monthly = mann_whitney_test(df, "MonthlyCharges")

print(f"Churned  group: n = {res_monthly['n_churned']},  "
      f"median monthly charges = ${res_monthly['median_churned']:.2f}")
print(f"Retained group: n = {res_monthly['n_retained']}, "
      f"median monthly charges = ${res_monthly['median_retained']:.2f}")
print(
    f"U = {res_monthly['statistic']:.4g} | "
    f"p = {res_monthly['p_value']:.4g} | "
    f"rank-biserial r = {res_monthly['effect_size']:.4f}"
)

plot_continuous_by_churn(
    df, "MonthlyCharges", "Monthly Charges Distribution by Churn Status"
)

Churned  group: n = 1869,  median monthly charges = $79.65
Retained group: n = 5163, median monthly charges = $64.45
U = 5.986e+06 | p = 8.467e-54 | rank-biserial r = -0.2407


### Result (computed from data)

Churned and retained customers had **significantly different monthly charge
distributions**:

| Measure | Value |
|---|---|
| Mann-Whitney U statistic | 5,986,149 |
| p-value | 8.47e-54 |
| Rank-biserial r | -0.241 |
| Median monthly charges, churned | $79.65 |
| Median monthly charges, retained | $64.45 |

**Plain-language interpretation.** Churned customers tend to pay more each
month (median $79.65 vs $64.45). The negative rank-biserial r (-0.24) means
churned customers tended to have *larger* monthly-charge values; the effect is
small-to-moderate. Higher monthly bills — often tied to fiber optic service
and add-on products — are associated with more churn.

**Association, not causation.** Monthly charges are not randomly assigned and
are entangled with service type, add-ons, and tenure. Customers may churn
because of price, or higher-spending customers may differ in other ways; this
test does not establish that price causes churn.

## 9. Hypothesis 4 — InternetService

**Business hypothesis.** Internet service type is associated with churn.

**Null hypothesis (H0).** InternetService and churn are independent.

**Why a chi-square test of independence?** Both variables are categorical. The
chi-square test checks whether churn rates differ systematically across DSL,
fiber optic, and no-internet customers. Cramér's V is reported as the effect
size.

In [10]:
res_internet, ct_internet = chi_square_test(df, "InternetService")

display(category_summary(df, "InternetService"))
print("\nContingency table (rows = InternetService, columns = Churn):")
display(ct_internet)

print(
    f"chi-square = {res_internet['statistic']:.4f} | "
    f"df = {res_internet['degrees_of_freedom']} | "
    f"p = {res_internet['p_value']:.4g} | "
    f"Cramer's V = {res_internet['effect_size']:.4f}"
)

plot_churn_rate_by_category(
    category_summary(df, "InternetService"),
    "InternetService",
    "Churn Rate by Internet Service Type",
)

,customers,churned,churn_rate
InternetService,,,
DSL,2416,459,0.189983
Fiber optic,3096,1297,0.418928
No,1520,113,0.074342



Contingency table (rows = InternetService, columns = Churn):


Churn,0,1
InternetService,,
DSL,1957,459
Fiber optic,1799,1297
No,1407,113


chi-square = 728.6956 | df = 2 | p = 5.831e-159 | Cramer's V = 0.3219


### Result (computed from data)

Internet service type was **statistically associated** with churn:

| Measure | Value |
|---|---|
| Chi-square statistic | 728.70 |
| Degrees of freedom | 2 |
| p-value | 5.83e-159 |
| Cramér's V | 0.322 |

Observed churn rates:

| InternetService | Customers | Churn rate |
|---|---|---|
| Fiber optic | 3,096 | 41.9% |
| DSL | 2,416 | 19.0% |
| No | 1,520 | 7.4% |

**Plain-language interpretation.** Fiber optic customers churn at about 2.2x
the DSL rate and more than 5x the rate of customers without internet service.
Cramér's V ≈ 0.32 is a moderate association.

**Association, not causation.** Fiber optic may attract a different customer
segment or a different price point; the observational design cannot isolate
whether the technology itself drives churn.

## 10. Hypothesis 5 — PaymentMethod

**Business hypothesis.** Payment method is associated with churn.

**Null hypothesis (H0).** PaymentMethod and churn are independent.

**Why a chi-square test of independence?** Both variables are categorical. The
test compares observed churn across the four payment methods with what would
be expected under independence. Cramér's V is reported as the effect size.

In [11]:
res_payment, ct_payment = chi_square_test(df, "PaymentMethod")

display(category_summary(df, "PaymentMethod"))
print("\nContingency table (rows = PaymentMethod, columns = Churn):")
display(ct_payment)

print(
    f"chi-square = {res_payment['statistic']:.4f} | "
    f"df = {res_payment['degrees_of_freedom']} | "
    f"p = {res_payment['p_value']:.4g} | "
    f"Cramer's V = {res_payment['effect_size']:.4f}"
)

plot_churn_rate_by_category(
    category_summary(df, "PaymentMethod"),
    "PaymentMethod",
    "Churn Rate by Payment Method",
)

,customers,churned,churn_rate
PaymentMethod,,,
Bank transfer (automatic),1542,258,0.167315
Credit card (automatic),1521,232,0.152531
Electronic check,2365,1071,0.452854
Mailed check,1604,308,0.192020



Contingency table (rows = PaymentMethod, columns = Churn):


Churn,0,1
PaymentMethod,,
Bank transfer (automatic),1284,258
Credit card (automatic),1289,232
Electronic check,1294,1071
Mailed check,1296,308


chi-square = 645.4299 | df = 3 | p = 1.426e-139 | Cramer's V = 0.3030


### Result (computed from data)

Payment method was **statistically associated** with churn:

| Measure | Value |
|---|---|
| Chi-square statistic | 645.43 |
| Degrees of freedom | 3 |
| p-value | 1.43e-139 |
| Cramér's V | 0.303 |

Observed churn rates:

| PaymentMethod | Customers | Churn rate |
|---|---|---|
| Electronic check | 2,365 | 45.3% |
| Mailed check | 1,604 | 19.2% |
| Bank transfer (automatic) | 1,542 | 16.7% |
| Credit card (automatic) | 1,521 | 15.3% |

**Plain-language interpretation.** Customers paying by electronic check churn
at roughly 3x the rate of automatic-payment customers. Automatic payment adds
a switching-cost friction, but it is also correlated with contract length and
other factors, so the interpretation must be cautious.

**Association, not causation.** Payment method is chosen by the customer and
correlates with contract type and tenure. The observational data cannot prove
that moving customers to automatic payment would reduce churn.

## 11. Hypothesis 6 — Customer Service / Security Features

Customers can buy add-on protection and support features. This section tests a
small, pre-selected group of related categorical variables — OnlineSecurity,
TechSupport, and OnlineBackup — rather than every column in the dataset.

**Business hypothesis (per feature).** Customers who have not purchased the
feature have a higher observed churn rate than customers who have.

**Null hypothesis (H0, per feature).** The feature and churn are independent.

**Why a chi-square test of independence?** Each feature is categorical and is
compared with the binary Churn target; chi-square tests independence and
Cramér's V measures the effect size.

In [12]:
security_features = ["OnlineSecurity", "TechSupport", "OnlineBackup"]
res_security_features = []

for feature in security_features:
    res, ct = chi_square_test(df, feature)
    res_security_features.append(res)
    display(category_summary(df, feature))
    print(
        f"{feature}: chi-square = {res['statistic']:.4f} | "
        f"df = {res['degrees_of_freedom']} | "
        f"p = {res['p_value']:.4g} | "
        f"Cramer's V = {res['effect_size']:.4f}"
    )

,customers,churned,churn_rate
OnlineSecurity,,,
No,3497,1461,0.417787
No internet service,1520,113,0.074342
Yes,2015,295,0.146402


OnlineSecurity: chi-square = 846.6774 | df = 2 | p = 1.401e-184 | Cramer's V = 0.3470


,customers,churned,churn_rate
TechSupport,,,
No,3472,1446,0.416475
No internet service,1520,113,0.074342
Yes,2040,310,0.151961


TechSupport: chi-square = 824.9256 | df = 2 | p = 7.408e-180 | Cramer's V = 0.3425


,customers,churned,churn_rate
OnlineBackup,,,
No,3087,1233,0.399417
No internet service,1520,113,0.074342
Yes,2425,523,0.215670


OnlineBackup: chi-square = 599.1752 | df = 2 | p = 7.776e-131 | Cramer's V = 0.2919


In [13]:
rows = []
for feature in security_features:
    summary = category_summary(df, feature)
    for category, row in summary.iterrows():
        rows.append({
            "Feature": feature,
            "Category": category,
            "Churn rate (%)": row["churn_rate"] * 100,
        })
rates_by_feature = pd.DataFrame(rows)

pivot = rates_by_feature.pivot(index="Category", columns="Feature",
                               values="Churn rate (%)")
ax = pivot.plot(kind="bar", figsize=(9, 4.5), width=0.8)
ax.set_ylabel("Churn rate (%)")
ax.set_xlabel("Feature status")
ax.set_title("Churn Rate by Service/Security Feature")
ax.legend(title="Feature")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

### Result (computed from data)

All three features were **statistically associated** with churn:

| Feature | Chi-square | df | p-value | Cramér's V | Churn rate without ("No") | Churn rate with ("Yes") |
|---|---|---|---|---|---|---|
| OnlineSecurity | 846.68 | 2 | 1.40e-184 | 0.347 | 41.8% | 14.6% |
| TechSupport | 824.93 | 2 | 7.41e-180 | 0.343 | 41.6% | 15.2% |
| OnlineBackup | 599.18 | 2 | 7.78e-131 | 0.292 | 39.9% | 21.6% |

**Plain-language interpretation.** Customers who have not purchased these
features churn at substantially higher rates than customers who have. The
"no internet service" group — the lowest-churn group in this dataset — is
always coded "No internet service" for these add-ons, so part of the gap
overlaps with the InternetService finding rather than being purely about the
add-on itself.

**Association, not causation.** Customers on different internet tiers may be
offered or choose add-ons differently. These results do not establish that
buying OnlineSecurity, TechSupport, or OnlineBackup *reduces* churn.

## 12. Multiple Hypothesis Testing

Eight statistical tests were performed in this notebook. Testing many
hypotheses inflates the chance of false positives: at alpha = 0.05, roughly
5% of true-null tests would be expected to appear significant purely by
chance.

The **Benjamini-Hochberg procedure** controls the **false discovery rate
(FDR)** — the expected proportion of "significant" findings that are false
positives — rather than the stricter family-wise error rate. Adjusted p-values
(q-values) are computed from the raw p-values, and a result is retained when
q < 0.05. This is why the adjusted p-values below are used instead of the raw
p-values when judging significance.

The table below lists every test performed, its raw p-value, its FDR-adjusted
p-value, and whether it remains significant.

In [14]:
all_results = [
    res_contract, res_internet, res_payment,
    *res_security_features,
    res_tenure, res_monthly,
]

summary_df = pd.DataFrame(all_results).drop(
    columns=["n_churned", "n_retained", "median_churned", "median_retained"],
    errors="ignore",
)

rejected, adjusted_p, _, _ = multipletests(
    summary_df["p_value"], alpha=0.05, method="fdr_bh"
)
summary_df["adjusted_p_value"] = adjusted_p
summary_df["significant_after_fdr"] = rejected

display(
    summary_df[
        ["feature", "test", "statistic", "degrees_of_freedom", "p_value",
         "adjusted_p_value", "effect_size", "significant_after_fdr"]
    ]
)
print(
    f"Tests performed: {len(summary_df)} | "
    f"Significant after FDR (alpha = 0.05): "
    f"{int(summary_df['significant_after_fdr'].sum())} of {len(summary_df)}"
)

,feature,test,statistic,degrees_of_freedom,p_value,adjusted_p_value,effect_size,significant_after_fdr
0,Contract,Chi-square,1.179546e+03,2.0,7.326182e-257,5.860946e-256,0.409560,True
1,InternetService,Chi-square,7.286956e+02,2.0,5.831199e-159,9.329918e-159,0.321909,True
2,PaymentMethod,Chi-square,6.454299e+02,3.0,1.426310e-139,1.901746e-139,0.302960,True
3,OnlineSecurity,Chi-square,8.466774e+02,2.0,1.400687e-184,3.735165e-184,0.346992,True
4,TechSupport,Chi-square,8.249256e+02,2.0,7.407808e-180,1.481562e-179,0.342506,True
5,OnlineBackup,Chi-square,5.991752e+02,2.0,7.776099e-131,8.886971e-131,0.291902,True
6,tenure,Mann-Whitney U,2.494979e+06,NaN,6.043047e-211,2.417219e-210,0.482887,True
7,MonthlyCharges,Mann-Whitney U,5.986148e+06,NaN,8.467195e-54,8.467195e-54,-0.240698,True


Tests performed: 8 | Significant after FDR (alpha = 0.05): 8 of 8


## 13. Final Hypothesis Summary

The table below consolidates every hypothesis. The **"Supported?"** column is
**computed** from the FDR-adjusted p-value (Yes if adjusted p < 0.05) — it is
never entered by hand.

In [15]:
hypothesis_labels = {
    "Contract": "H1 Contract type",
    "InternetService": "H4 Internet service type",
    "PaymentMethod": "H5 Payment method",
    "OnlineSecurity": "H6 Online security",
    "TechSupport": "H6 Tech support",
    "OnlineBackup": "H6 Online backup",
    "tenure": "H2 Tenure",
    "MonthlyCharges": "H3 Monthly charges",
}

final_summary = summary_df.copy()
final_summary["Hypothesis"] = final_summary["feature"].map(hypothesis_labels)
final_summary["Supported?"] = np.where(
    final_summary["significant_after_fdr"], "Yes", "No"
)

display(
    final_summary[
        ["Hypothesis", "feature", "test", "statistic", "degrees_of_freedom",
         "p_value", "adjusted_p_value", "effect_size", "Supported?"]
    ].rename(columns={
        "feature": "Feature",
        "test": "Test",
        "statistic": "Statistic",
        "degrees_of_freedom": "df",
        "p_value": "Raw p-value",
        "adjusted_p_value": "FDR-adjusted p-value",
        "effect_size": "Effect size",
    })
)

,Hypothesis,Feature,Test,Statistic,df,Raw p-value,FDR-adjusted p-value,Effect size,Supported?
0,H1 Contract type,Contract,Chi-square,1.179546e+03,2.0,7.326182e-257,5.860946e-256,0.409560,Yes
1,H4 Internet service type,InternetService,Chi-square,7.286956e+02,2.0,5.831199e-159,9.329918e-159,0.321909,Yes
2,H5 Payment method,PaymentMethod,Chi-square,6.454299e+02,3.0,1.426310e-139,1.901746e-139,0.302960,Yes
3,H6 Online security,OnlineSecurity,Chi-square,8.466774e+02,2.0,1.400687e-184,3.735165e-184,0.346992,Yes
4,H6 Tech support,TechSupport,Chi-square,8.249256e+02,2.0,7.407808e-180,1.481562e-179,0.342506,Yes
5,H6 Online backup,OnlineBackup,Chi-square,5.991752e+02,2.0,7.776099e-131,8.886971e-131,0.291902,Yes
6,H2 Tenure,tenure,Mann-Whitney U,2.494979e+06,NaN,6.043047e-211,2.417219e-210,0.482887,Yes
7,H3 Monthly charges,MonthlyCharges,Mann-Whitney U,5.986148e+06,NaN,8.467195e-54,8.467195e-54,-0.240698,Yes


## 14. Modeling Implications (for Notebook 03)

This EDA was designed to inform — not dictate — the modeling work. Key
takeaways to carry forward:

- **Association is not causation.** No feature above was shown to *cause*
  churn. These features are candidates for predictive modelling, not proven
  drivers.
- **The target is imbalanced.** 26.6% of customers churned. Notebook 03 must
  handle class imbalance explicitly and evaluate models with metrics other
  than accuracy (e.g., precision/recall, AUROC/AUPRC).
- **Statistical significance is not predictive importance.** A tiny p-value
  means the observed association is unlikely under the null; it says nothing
  about how much a feature improves out-of-sample prediction.
- **Effect size matters.** Features with larger effect sizes (Contract,
  OnlineSecurity/TechSupport, tenure) are promising candidates, but their
  predictive contribution must be re-evaluated within a model.
- **Candidate features.** Contract, InternetService, PaymentMethod,
  OnlineSecurity, TechSupport, and OnlineBackup are strong categorical
  candidates; tenure and MonthlyCharges are continuous candidates.
- **No modeling in this notebook.** Notebook 03 will split the data into
  train/dev/test, select models and thresholds using the train/dev portions,
  and evaluate predictive performance separately.
- **Holdout integrity.** The final holdout set must remain untouched until
  the final evaluation in Notebook 03 so that reported performance is unbiased.

## 15. Limitations

This analysis has genuine limitations and should be read accordingly:

- **Observational data.** The dataset is observational; customers were not
  randomly assigned to contract types, services, or prices.
- **No causal inference.** Statistical association does not establish
  causality. Confounders (e.g., tenure and contract type are strongly related)
  may drive the associations observed here.
- **Single dataset, single period.** Findings are based on 7,032 customers at
  a single point in time and may not generalize to another provider, market,
  period, or customer population.
- **No predictive claims.** Hypothesis testing measures association, not
  predictive performance. Whether these features improve prediction is an open
  question for Notebook 03.
- **Multiple testing.** Eight tests were performed. The Benjamini-Hochberg
  correction was applied, but no correction removes unmeasured confounding.
- **Test assumptions.** Chi-square assumes adequate expected cell counts; all
  contingency tables here had far more than 5 expected counts per cell, so
  this assumption is comfortably met. Mann-Whitney U assumes independent
  observations; each row is a distinct customer, so this holds.
- **Effect-size thresholds** (small / medium / large for Cramér's V and the
  rank-biserial correlation) are heuristic conventions from the literature,
  not mathematical guarantees.